In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
!pip -q install duckdb pandas pyarrow fsspec huggingface_hub

In [3]:
import duckdb

con = duckdb.connect()

In [4]:
from huggingface_hub import hf_hub_download

TABLE = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)

print(TABLE)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [5]:
query = f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{TABLE}')
"""

con.sql(query).df()

,rows,start_date,end_date
0,11694072,2026-06-01,2026-06-30


In [6]:
query = f"""
SELECT *
FROM read_parquet('{TABLE}')
LIMIT 5;
"""

df = con.sql(query).df()

df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [7]:
query = f"""
DESCRIBE
SELECT *
FROM read_parquet('{TABLE}');
"""

con.sql(query).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ni7chal14/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
from IPython.display import Markdown, display

summary = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{TABLE}')
""").df()

rows = int(summary.loc[0, "rows"])
start_date = summary.loc[0, "start_date"]
end_date = summary.loc[0, "end_date"]

display(Markdown(f"""
# Unit of Analysis

**Intended Unit of Analysis**

One row represents the daily performance of one content item for one client on one report date.

**Intended Grain**

(report_date, client_hash_id, content_hash_id)

**Time Window**

- Start Date: {start_date.date()}
- End Date: {end_date.date()}
- Total Rows: {rows:,}
"""))


# Unit of Analysis

**Intended Unit of Analysis**

One row represents the daily performance of one content item for one client on one report date.

**Intended Grain**

(report_date, client_hash_id, content_hash_id)

**Time Window**

- Start Date: 2026-06-01
- End Date: 2026-06-30
- Total Rows: 11,694,072


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
from IPython.display import Markdown, display

display(Markdown("""
# Field Classification

## Features
The following columns are suitable as model features because they describe search, traffic, and engagement metrics.

- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

## Label
No prediction label exists in this table. A target variable will need to be defined for any downstream machine learning task.

## Context
These fields identify or organize the data but should not be used as predictive features.

- report_date
- client_hash_id
- content_hash_id
- month

## Excluded
The following fields should not be used as model features because they uniquely identify entities.

- client_hash_id
- content_hash_id
"""))


# Field Classification

## Features
The following columns are suitable as model features because they describe search, traffic, and engagement metrics.

- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

## Label
No prediction label exists in this table. A target variable will need to be defined for any downstream machine learning task.

## Context
These fields identify or organize the data but should not be used as predictive features.

- report_date
- client_hash_id
- content_hash_id
- month

## Excluded
The following fields should not be used as model features because they uniquely identify entities.

- client_hash_id
- content_hash_id


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM read_parquet('{TABLE}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count
0,2026-06-13,client_1a730cb2640a1abf,content_0a21add649629840,2
1,2026-06-13,client_1a730cb2640a1abf,content_0ffd552a735a66ab,2
2,2026-06-13,client_1a730cb2640a1abf,content_f85822ce54f90378,2
3,2026-06-13,client_1a730cb2640a1abf,content_ff51845ed161bfb4,2
4,2026-06-13,client_1a730cb2640a1abf,content_8f5be8e45452dda7,2
5,2026-06-13,client_1a730cb2640a1abf,content_a4990373223aa784,2
6,2026-06-13,client_1a730cb2640a1abf,content_b5aec9a8a2ee7fb0,2
7,2026-06-13,client_1a730cb2640a1abf,content_c426957b8369a333,2
8,2026-06-13,client_1a730cb2640a1abf,content_5ea66e7cd969a440,2
9,2026-06-13,client_1a730cb2640a1abf,content_62a6c7628cfd006f,2


In [11]:
summary_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{TABLE}');
""").df()

summary_check

,total_rows,start_date,end_date
0,11694072,2026-06-01,2026-06-30


In [12]:
missing_check = con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS gsc_avg_position_missing,
    AVG(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS ga4_pageviews_missing,
    AVG(CASE WHEN sessions_ai IS NULL THEN 1 ELSE 0 END) AS sessions_ai_missing
FROM read_parquet('{TABLE}');
""").df()

missing_check

,gsc_avg_position_missing,ga4_pageviews_missing,sessions_ai_missing
0,0.668302,0.205012,0.205012


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
from IPython.display import Markdown, display

display(Markdown(f"""
# Data Limits

- This sample dataset only contains data from **{summary_check.loc[0, 'start_date']}** to **{summary_check.loc[0, 'end_date']}** and does not represent the complete warehouse history.
- The intended grain is `(report_date, client_hash_id, content_hash_id)`, but the verification query found duplicate records for some key combinations. These should be investigated or removed before modeling.
- Several columns contain substantial missing values:
  - `gsc_avg_position`: {missing_check.loc[0, 'gsc_avg_position_missing']:.1%} missing
  - `ga4_pageviews`: {missing_check.loc[0, 'ga4_pageviews_missing']:.1%} missing
  - `sessions_ai`: {missing_check.loc[0, 'sessions_ai_missing']:.1%} missing
- Client and content identifiers are anonymized, so they cannot be interpreted directly.
- Availability flags (`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`) should be considered when analyzing performance metrics.
"""))


# Data Limits

- This sample dataset only contains data from **2026-06-01 00:00:00** to **2026-06-30 00:00:00** and does not represent the complete warehouse history.
- The intended grain is `(report_date, client_hash_id, content_hash_id)`, but the verification query found duplicate records for some key combinations. These should be investigated or removed before modeling.
- Several columns contain substantial missing values:
  - `gsc_avg_position`: 66.8% missing
  - `ga4_pageviews`: 20.5% missing
  - `sessions_ai`: 20.5% missing
- Client and content identifiers are anonymized, so they cannot be interpreted directly.
- Availability flags (`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`) should be considered when analyzing performance metrics.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.